# DQN Agent Training and Testing for Stock Portfolio Management

This notebook implements a Deep Q-Network (DQN) agent for automated stock trading with rolling window backtesting strategy. The agent is trained on DOW-30 stocks with technical indicators and tested against the Dow Jones Index benchmark.

## Overview
- **Data**: DOW-30 stocks from Yahoo Finance
- **Agent**: Deep Q-Network (DQN) via ElegantRL
- **Environment**: StockTradingEnv with transaction costs (discretized action space)
- **Validation**: Rolling window approach (252-day training, 20-day validation/test periods)
- **Benchmark**: Dow Jones Index

In [44]:
import os
import pandas as pd
import numpy as np
import torch
import yfinance as yf
import gymnasium as gym
import warnings

warnings.filterwarnings('ignore')

# Patch torch.load to allow loading models without weights_only=True
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return torch._original_load(*args, **kwargs)

torch.load = patched_torch_load

# Patch yf.download to remove proxy parameter
if not hasattr(yf, '_original_download'):
    yf._original_download = yf.download

def patched_download(*args, **kwargs):
    kwargs.pop('proxy', None)
    return yf._original_download(*args, **kwargs)

yf.download = patched_download

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from elegantrl.agents import AgentDQN
from elegantrl.train.run import train_agent

try:
    from elegantrl.train.config import Config as Arguments
except ImportError:
    from elegantrl.train.config import Arguments

print("[✓] All imports successful!")

[✓] All imports successful!


In [45]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.preprocessors import FeatureEngineer

def prepare_data(cache_file = "../data/stock_dow30_cache.csv",
        TRAIN_START_DATE = '2010-01-01',
        TEST_END_DATE = '2023-12-30'):
    """
    Load stock data from cache or download from Yahoo Finance.
    Returns preprocessed dataframe with technical indicators.
    """
    
    # Create data directory if it doesn't exist
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)
    
    if os.path.exists(cache_file):
        print(f"[INFO] Loading validated cache '{cache_file}'...")
        try:
            df = pd.read_csv(cache_file)
            df = df.drop_duplicates(subset=['date', 'tic'])
            if len(df) > 0:
                print(f"[✓] Cache loaded: {len(df)} records, Date range: {df['date'].min()} to {df['date'].max()}")
                return df
            else:
                print("[!] Cache file is empty, will re-download...")
        except Exception as e:
            print(f"[!] Error loading cache: {e}, will re-download...")

    
    else:
        df = YahooDownloader(start_date=TRAIN_START_DATE,
                            end_date=TEST_END_DATE,
                            ticker_list=DOW_30_TICKER).fetch_data()


        print(f"[✓] Data preparation complete!")
        print(f"    - Total records: {len(df)}")
        print(f"    - Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"    - Unique tickers: {df['tic'].nunique()}")

        # Add technical indicators
        print("\n[STEP 2.5] Calculating technical indicators...")

        # Define indicators to calculate
        TECHNICAL_INDICATORS = ["macd", "boll_ub", "boll_lb", "rsi_30", "cci_30", "dx_30", "close_30_sma", "close_60_sma"]

        try:
            fe = FeatureEngineer(
                use_technical_indicator=True,
                tech_indicator_list=TECHNICAL_INDICATORS,
                use_vix=False,
                use_turbulence=False,
                user_defined_feature=False
            )
            
            df = fe.preprocess_data(df)
            
            # Rename close_30_sma and close_60_sma to match expected names
            if 'close_30_sma' in df.columns:
                df = df.rename(columns={'close_30_sma': 'close_30'})
            if 'close_60_sma' in df.columns:
                df = df.rename(columns={'close_60_sma': 'close_60'})
            
            print(f"[✓] Technical indicators calculated!")
            print(f"    Available columns: {df.columns.tolist()}")
            
        except Exception as e:
            print(f"[!] Error calculating technical indicators: {e}")
            print(f"[!] Attempting alternative method...")
            
            # Alternative: Calculate indicators manually using ta library
            import ta
            
            df_list = []
            for ticker in df['tic'].unique():
                ticker_df = df[df['tic'] == ticker].copy()
                ticker_df = ticker_df.sort_values('date')
                
                # Calculate indicators
                ticker_df['macd'] = ta.trend.macd_diff(ticker_df['close'])
                
                bb = ta.volatility.BollingerBands(ticker_df['close'])
                ticker_df['boll_ub'] = bb.bollinger_hband()
                ticker_df['boll_lb'] = bb.bollinger_lband()
                
                ticker_df['rsi_30'] = ta.momentum.rsi(ticker_df['close'], window=30)
                ticker_df['cci_30'] = ta.trend.cci(ticker_df['high'], ticker_df['low'], ticker_df['close'], window=30)
                ticker_df['dx_30'] = ta.trend.adx(ticker_df['high'], ticker_df['low'], ticker_df['close'], window=30)
                ticker_df['close_30'] = ticker_df['close'].rolling(window=30).mean()
                ticker_df['close_60'] = ticker_df['close'].rolling(window=60).mean()
                
                df_list.append(ticker_df)
            
            df = pd.concat(df_list, ignore_index=True)
            df = df.dropna()  # Remove rows with NaN from indicator calculation
            
            print(f"[✓] Technical indicators calculated using ta library!")

        if df is not None and not df.empty:
            df.to_csv(cache_file, index=False)
            print(f"[✓] Data saved to cache: {cache_file}")
        else:
            print("[ERROR] Failed to download data or received empty dataframe")
            return None
    
    return df

def split_train_test(df,
            TRAIN_START_DATE = '2010-01-07',
            TRAIN_END_DATE = '2023-10-24',
            TEST_START_DATE = '2023-10-25',
            TEST_END_DATE = '2023-11-21'):

    df['date'] = pd.to_datetime(df['date'])

    df_train = df[
        (df['date'] >= TRAIN_START_DATE) &
        (df['date'] <= TRAIN_END_DATE)
    ].copy()

    df_test = df[
        (df['date'] >= TEST_START_DATE) &
        (df['date'] <= TEST_END_DATE)
    ].copy()

    print(f"[INFO] Train: {df_train['date'].min()} → {df_train['date'].max()}, "
          f"{len(df_train)} rows")

    print(f"[INFO] Test:  {df_test['date'].min()} → {df_test['date'].max()}, "
          f"{len(df_test)} rows")

    return df_train, df_test

# Load data
print("\n[STEP 1] Loading stock data...")
df = prepare_data()
df_train, df_test = split_train_test(df)


[STEP 1] Loading stock data...
[INFO] Loading validated cache '../data/stock_dow30_cache.csv'...
[✓] Cache loaded: 98616 records, Date range: 2010-01-04 to 2023-12-29
[INFO] Train: 2010-01-07 00:00:00 → 2023-10-24 00:00:00, 97244 rows
[INFO] Test:  2023-10-25 00:00:00 → 2023-11-21 00:00:00, 560 rows


In [46]:
print(df_train.head())
print(df_test.head())
print(df_test.columns.tolist())

         date      close       high        low       open     volume   tic  \
84 2010-01-07   6.309609   6.352157   6.263766   6.344667  477131200  AAPL   
85 2010-01-07  38.060387  38.236247  36.964638  38.155081   10371600  AMGN   
86 2010-01-07  33.469982  33.677278  32.776346  32.895937    8981700   AXP   
87 2010-01-07  48.468552  48.554268  45.990577  46.372402   14379100    BA   
88 2010-01-07  39.948700  40.102686  39.265818  39.700988    5432900   CAT   

    day      macd    boll_ub    boll_lb      rsi_30      cci_30       dx_30  \
84    3 -0.004614   6.485744   6.247634    8.573653 -112.341813   61.789070   
85    3 -0.035977  39.389457  37.718864    0.000000 -133.333333  100.000000   
86    3  0.040332  33.802108  31.848378   93.970753  105.083585  100.000000   
87    3  0.172910  50.009109  42.011009  100.000000  103.114519  100.000000   
88    3  0.025526  40.295856  39.005706  100.000000   60.985770   25.730463   

     close_30   close_60  
84   6.366689   6.366689  
85

In [47]:
# Training and Testing with Full Dataset
ALL_INDICATORS = ["macd", "boll_ub", "boll_lb", "rsi_30", "cci_30", "dx_30", "close_30", "close_60"]

# Prepare training data
stock_dimension = len(df_train['tic'].unique()) # Number of unique stocks in training data
state_dim = 1 + stock_dimension * (len(ALL_INDICATORS) + 2) # State dimension: cash + (stock_dim * (indicators + price + shares))

# Create day indices for training data
train_dates = sorted(df_train['date'].unique())
date_map_train = {date: idx for idx, date in enumerate(train_dates)}
df_train['day'] = df_train['date'].map(date_map_train) # Map dates to day indices for training data
df_train.set_index('day', inplace=True, drop=False)

# Create day indices for test data
test_dates = sorted(df_test['date'].unique())
date_map_test = {date: idx for idx, date in enumerate(test_dates)}
df_test['day'] = df_test['date'].map(date_map_test) # Map dates to day indices for test data
df_test.set_index('day', inplace=True, drop=False)

print(f"[INFO] Stock dimension: {stock_dimension}")
print(f"[INFO] State dimension: {state_dim}")
print(f"[INFO] Training data shape: {df_train.shape}")
print(f"[INFO] Test data shape: {df_test.shape}")

[INFO] Stock dimension: 28
[INFO] State dimension: 281
[INFO] Training data shape: (97244, 16)
[INFO] Test data shape: (560, 16)


In [48]:
from env_wrapper.dqn_env_wrapper import ElegantDQNWrapper
def setup_dqn_args(env_args, cwd_path):
    args = Arguments(agent_class=AgentDQN, env_class=ElegantDQNWrapper)
    args.env_args = env_args
    args.env_name = env_args['env_name']

    # DQN 网络配置
    args.net_dims = (128, 128, 128)  # 三层全连接网络
    args.state_dim = env_args['state_dim']
    args.action_dim = env_args['action_dim']
    args.if_discrete = True  # DQN 使用离散动作空间

    args.learning_rate = 1e-4
    args.batch_size = 256
    args.explore_rate = 0.01  # Epsilon-greedy 探索率

    args.target_step = 2000
    args.break_step = 100000

    # Training progress and logging
    args.worker_num = 1
    args.eval_proc_num = 0
    args.eval_gap = 500
    args.save_gap = 200
    
    args.cwd = cwd_path
    args.if_remove = False
    return args

print("[✓] setup_dqn_args function defined!")

[✓] setup_dqn_args function defined!


In [61]:
def real_test_inference(test_df, stock_dim, indicators, args, env_params):
    env = ElegantDQNWrapper(**env_params)

    agent = AgentDQN(args.net_dims, args.state_dim, args.action_dim)
    agent.save_or_load_agent(args.cwd, if_save=False)
    agent.act.eval()
    agent.explore_rate = 0.0  # 测试时不探索

    res = env.reset()
    state = res[0] if isinstance(res, tuple) else res

    done = False
    while not done:
        s_tensor = torch.as_tensor((state,), dtype=torch.float32, device=agent.device)
        with torch.no_grad():
            q_values = agent.act.get_q_value(s_tensor)
            print(q_values)
            action = q_values.argmax(dim=1).item()
            # print(action)
        
        step_res = env.step(action)
        if len(step_res) == 5:
            state, reward, term, trunc, _ = step_res
            done = term or trunc
        else:
            state, reward, done, _, _ = step_res

    return env.env.save_asset_memory()

print("[✓] real_test_inference function defined!")

[✓] real_test_inference function defined!


In [ ]:
# Setup environment parameters for training
discrete_action_dim = stock_dimension * 3 + 1  # DQN 离散动作空间 # 每只股票有3个动作（买、卖、持有），加上一个整体不操作的动作


env_params = {
    "env_name": "FinRL_DQN_Train",
    "df": df_train,
    "stock_dim": stock_dimension,
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": [0.001] * stock_dimension,
    "sell_cost_pct": [0.001] * stock_dimension,
    "reward_scaling": 1e-4,
    "state_space": state_dim,
    "action_space": discrete_action_dim,
    "tech_indicator_list": ALL_INDICATORS,
    "state_dim": state_dim,
    "action_dim": discrete_action_dim,
    "if_discrete": True,
    "target_return": 10.0
}

print(f"\n[STEP 6] Starting DQN Agent Training on Full Dataset...")
print(f"Training period: {df_train['date'].min()} to {df_train['date'].max()}")
print(f"Test period: {df_test['date'].min()} to {df_test['date'].max()}")
print("\n" + "="*80)

try:
    cwd_path = "./checkpoints/stock/dqn_full_train"
    os.makedirs(cwd_path, exist_ok=True)

    args = setup_dqn_args(env_params, cwd_path)

    print(f"\n[Training Configuration]")
    print(f"  Network Dims: {args.net_dims}")
    print(f"  Learning Rate: {args.learning_rate}")
    print(f"  Batch Size: {args.batch_size}")
    print(f"  Explore Rate: {args.explore_rate}")
    print(f"  Target Steps per Update: {args.target_step}")
    print(f"  Total Training Steps: {args.break_step}")
    print(f"  Action Space: {discrete_action_dim} (discrete)")
    print(f"  Checkpoint Path: {cwd_path}")
    print("="*80 + "\n")

    print("[-->] Starting DQN Agent Training...")
    
    train_agent(args)
    
    print("\n" + "="*80)
    print("[✓] Training completed successfully!")
    print("="*80)

    # Run inference on test data
    print(f"\n[STEP 6.5] Running inference on test data...")
    
    # Update env_params for test
    test_env_params = env_params.copy()
    test_env_params['df'] = df_test
    test_env_params['env_name'] = "FinRL_DQN_Test"
    
    test_results = real_test_inference(df_test, stock_dimension, ALL_INDICATORS, args, test_env_params)
    # print("stock_dimension:", stock_dimension)
    log_path = "./logs/stock/"
    os.makedirs(log_path, exist_ok=True)
    # Save test results
    test_results.to_csv(log_path + "dqn_test_results.csv", index=False)
    print(f"[✓] Test results saved to '{log_path + 'dqn_test_results.csv'}'")
    print(f"    Records: {len(test_results)}")
    print(f"    Date range: {test_results['date'].min()} to {test_results['date'].max()}")
    
    # Calculate and display initial performance metrics
    initial_value = test_results['account_value'].iloc[0]
    final_value = test_results['account_value'].iloc[-1]
    total_return = (final_value / initial_value - 1) * 100
    
    print(f"    Initial Portfolio Value: ${initial_value:,.2f}")
    print(f"    Final Portfolio Value: ${final_value:,.2f}")
    print(f"    Total Return: {total_return:.2f}%")

except Exception as e:
    print(f"[ERROR] Training failed: {e}")
    import traceback
    traceback.print_exc()


[STEP 6] Starting DQN Agent Training on Full Dataset...
Training period: 2010-01-07 00:00:00 to 2023-10-24 00:00:00
Test period: 2023-10-25 00:00:00 to 2023-11-21 00:00:00


[Training Configuration]
  Network Dims: (128, 128, 128)
  Learning Rate: 0.0001
  Batch Size: 256
  Explore Rate: 0.01
  Target Steps per Update: 2000
  Total Training Steps: 100000
  Action Space: 85 (discrete)
  Checkpoint Path: ./checkpoints/stock/dqn_full_train

[-->] Starting DQN Agent Training...

[✓] Training completed successfully!

[STEP 6.5] Running inference on test data...
tensor([[3725.5662, 3789.4233, 3721.1619, 3157.7227, 3772.0061, 3717.3169,
         3740.6875, 3753.2188, 3633.9016, 3608.6831, 3781.6946, 3723.4521,
         3361.6465, 3710.2664, 3749.3225, 3170.9805, 3857.1736, 3696.9661,
         3497.0950, 3789.2830, 3768.2517, 3539.2712, 3780.2151, 3751.3645,
         3653.3464, 3711.4822, 3730.1321, 3835.4790, 3750.4707, 3714.2332,
         3894.6951, 3798.2871, 3713.9214, 3700.9136, 3667.276

In [53]:
import matplotlib.pyplot as plt

def calculate_metrics(df, column_name='account_value'):
    """Calculate key performance metrics"""
    daily_return = df[column_name].pct_change().dropna()

    cum_return = (df[column_name].iloc[-1] / df[column_name].iloc[0]) - 1

    if daily_return.std() != 0:
        sharpe_ratio = (252 ** 0.5) * (daily_return.mean() / daily_return.std())
    else:
        sharpe_ratio = 0

    rolling_max = df[column_name].cummax()
    drawdown = (df[column_name] - rolling_max) / rolling_max
    max_drawdown = drawdown.min()

    return cum_return, sharpe_ratio, max_drawdown


def dqn_test(log_path="./logs/stock/", result_file="dqn_test_results.csv"):
    """Test DQN strategy performance and compare with benchmark"""
    result_file = os.path.join(log_path, result_file)

    try:
        df = pd.read_csv(result_file)
        df['date'] = pd.to_datetime(df['date'])
        df = df.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)

        start_date = df['date'].iloc[0].strftime('%Y-%m-%d')
        end_date = df['date'].iloc[-1].strftime('%Y-%m-%d')
        initial_capital = df['account_value'].iloc[0]

        print(f"\n[STEP 7] Performance Analysis")
        print(f"[*] Successfully loaded DQN trading records!")
        print(f"    Testing period: {start_date} to {end_date}")
        print(f"    Test duration: {len(df)} trading days")
        print(f"[*] Downloading Dow Jones Index as benchmark...")

        # Download benchmark data
        benchmark = yf.download("^DJI", start=start_date, end=end_date, progress=False)

        if isinstance(benchmark.columns, pd.MultiIndex):
            benchmark.columns = benchmark.columns.get_level_values(0)

        benchmark = benchmark.reset_index()

        # Unify date column name
        if 'Date' not in benchmark.columns and 'index' in benchmark.columns:
            benchmark = benchmark.rename(columns={'index': 'Date'})

        benchmark['Date'] = pd.to_datetime(benchmark['Date'])

        # Merge tables
        df = pd.merge(df, benchmark[['Date', 'Close']], left_on='date', right_on='Date', how='left')
        df['Close'] = df['Close'].ffill()  # Fill missing weekend/holiday data

        df['benchmark_value'] = (df['Close'] / df['Close'].iloc[0]) * initial_capital

        # Calculate metrics
        dqn_ret, dqn_sharpe, dqn_mdd = calculate_metrics(df, 'account_value')
        bm_ret, bm_sharpe, bm_mdd = calculate_metrics(df, 'benchmark_value')

        print("\n" + "=" * 80)
        print(f"{'🚀 DQN Agent Stock Trading Performance Report 🚀':^80}")
        print("=" * 80)
        print(f"{'Metric':<34} {'DQN Strategy':>22} {'Benchmark':>22}")
        print("-" * 80)
        print(f"{'Cumulative Return':<34} {dqn_ret*100:>21.2f}% {bm_ret*100:>21.2f}%")
        print(f"{'Annualized Sharpe Ratio':<34} {dqn_sharpe:>22.4f} {bm_sharpe:>22.4f}")
        print(f"{'Maximum Drawdown':<34} {dqn_mdd*100:>21.2f}% {bm_mdd*100:>21.2f}%")
        print("=" * 80)

        # Calculate outperformance
        excess_return = (dqn_ret - bm_ret) * 100
        sharpe_diff = dqn_sharpe - bm_sharpe
        dd_improvement = (bm_mdd - dqn_mdd) * 100

        print(f"{'Excess Return':<34} {excess_return:>21.2f}%")
        print(f"{'Sharpe Ratio Difference':<34} {sharpe_diff:>22.4f}")
        print(f"{'Max Drawdown Improvement':<34} {dd_improvement:>21.2f}%")
        print("=" * 80 + "\n")

        # Create visualization
        plt.figure(figsize=(15, 8))
        plt.style.use('seaborn-v0_8-darkgrid')

        plt.plot(df['date'], df['account_value'], label='DQN Agent Portfolio', 
                color='#2E86AB', linewidth=2.5)
        plt.plot(df['date'], df['benchmark_value'], label='Dow Jones Index (Benchmark)', 
                color='#A23B72', linewidth=1.5, linestyle='--', alpha=0.8)

        plt.title('Deep Q-Network (DQN) Stock Trading Performance', 
                 fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Date', fontsize=12, fontweight='bold')
        plt.ylabel('Portfolio Value ($)', fontsize=12, fontweight='bold')
        plt.grid(True, alpha=0.3)

        textstr = '\n'.join((
            r'$\bf{DQN\ Strategy}$',
            f'Return: {dqn_ret * 100:.2f}%',
            f'Sharpe: {dqn_sharpe:.3f}',
            f'Max DD: {dqn_mdd * 100:.2f}%',
            '',
            r'$\bf{Benchmark}$',
            f'Return: {bm_ret * 100:.2f}%',
            f'Sharpe: {bm_sharpe:.3f}',
            f'Max DD: {bm_mdd * 100:.2f}%',
            '',
            r'$\bf{Outperformance}$',
            f'Excess Return: {excess_return:.2f}%',
            f'Sharpe Diff: {sharpe_diff:.3f}'
        ))
        props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray', linewidth=1.5)
        plt.gca().text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=9.5,
                       verticalalignment='top', bbox=props, family='monospace')

        plt.legend(loc='lower right', fontsize=11, framealpha=0.95)
        plt.tight_layout()

        # Save figure
        fig_path = os.path.join(log_path, 'dqn_backtest_performance.png')
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"[✓] Equity curve visualization saved to '{fig_path}'!")
        
        plt.show()

    except Exception as e:
        print(f"[ERROR] An error occurred during testing: {e}")
        import traceback
        traceback.print_exc()


# Run DQN test
dqn_test()
print("[✓] DQN Test Complete!")


[STEP 7] Performance Analysis
[*] Successfully loaded DQN trading records!
    Testing period: 2023-10-25 to 2023-11-21
    Test duration: 20 trading days
[*] Downloading Dow Jones Index as benchmark...
YF.download() has changed argument auto_adjust default to True

                 🚀 DQN Agent Stock Trading Performance Report 🚀                 
Metric                                       DQN Strategy              Benchmark
--------------------------------------------------------------------------------
Cumulative Return                                  13.44%                  6.40%
Annualized Sharpe Ratio                            8.5269                 6.8279
Maximum Drawdown                                   -0.86%                 -1.87%
Excess Return                                       7.03%
Sharpe Ratio Difference                            1.6990
Max Drawdown Improvement                           -1.01%

[✓] Equity curve visualization saved to './logs/stock/dqn_backtest_perf